In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set working directory
import os
os.chdir('/content/drive/MyDrive/brain-tumor-project/notebooks')

# Verify
!ls ../data/classification/train/

# 00 - Dataset Download & Merge Pipeline

Download 3 Kaggle datasets, deduplicate using perceptual hashing, and merge into ~15,000+ unique images.

**Datasets:**
1. `sartajbhuvaji/brain-tumor-classification-mri` (~3,264 images)
2. `masoudnickparvar/brain-tumor-mri-dataset` (~7,153 images)
3. `ahmedhamada0/brain-tumor-detection` Br35H (~3,060 images)

In [ ]:
# Install dependencies
!pip install kaggle imagehash Pillow scikit-learn tqdm -q

## 1. Setup Paths

In [ ]:
import os
import sys

BASE_DIR = os.path.dirname(os.getcwd())
RAW_DIR = os.path.join(BASE_DIR, "data", "raw")
MERGED_DIR = os.path.join(BASE_DIR, "data", "classification")
os.makedirs(RAW_DIR, exist_ok=True)
print(f"Raw data directory: {RAW_DIR}")
print(f"Merged data directory: {MERGED_DIR}")

## 2. Download Datasets

> **Note:** You need a Kaggle API key. Place `kaggle.json` in `~/.kaggle/`

In [ ]:
# Dataset 1: Sartaj
ds1_dir = os.path.join(RAW_DIR, "dataset1_sartaj")
if not os.path.exists(ds1_dir) or len(os.listdir(ds1_dir)) == 0:
    os.makedirs(ds1_dir, exist_ok=True)
    !kaggle datasets download -d sartajbhuvaji/brain-tumor-classification-mri -p {ds1_dir} --unzip
    print("Dataset 1 downloaded!")
else:
    print("Dataset 1 already exists.")

In [ ]:
# Dataset 2: Masoud
ds2_dir = os.path.join(RAW_DIR, "dataset2_masoud")
if not os.path.exists(ds2_dir) or len(os.listdir(ds2_dir)) == 0:
    os.makedirs(ds2_dir, exist_ok=True)
    !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset -p {ds2_dir} --unzip
    print("Dataset 2 downloaded!")
else:
    print("Dataset 2 already exists.")

In [ ]:
# Dataset 3: Br35H
ds3_dir = os.path.join(RAW_DIR, "dataset3_br35h")
if not os.path.exists(ds3_dir) or len(os.listdir(ds3_dir)) == 0:
    os.makedirs(ds3_dir, exist_ok=True)
    !kaggle datasets download -d ahmedhamada0/brain-tumor-detection -p {ds3_dir} --unzip
    print("Dataset 3 downloaded!")
else:
    print("Dataset 3 already exists.")

## 3. Inspect Raw Datasets

In [ ]:
sys.path.append(BASE_DIR)
from utils.preprocessing import scan_dataset_folder, CLASSES

for ds_name, ds_path in [("Dataset 1 (Sartaj)", ds1_dir), ("Dataset 2 (Masoud)", ds2_dir), ("Dataset 3 (Br35H)", ds3_dir)]:
    print(f"\n{'='*50}")
    print(f"{ds_name}: {ds_path}")
    print(f"{'='*50}")
    images = scan_dataset_folder(ds_path)
    for cls, paths in images.items():
        print(f"  {cls}: {len(paths)} images")
    print(f"  Total: {sum(len(p) for p in images.values())} images")

## 4. Merge & Deduplicate Datasets

In [ ]:
from utils.preprocessing import merge_datasets

print("Starting merge and deduplication...")
print("This may take several minutes...\n")

stats = merge_datasets(
    dataset_dirs=[ds1_dir, ds2_dir, ds3_dir],
    output_dir=MERGED_DIR,
    size=224,
    use_phash=True,
    hash_size=16
)

print(f"\n{'='*60}")
print("MERGE COMPLETE")
print(f"{'='*60}")
print(f"Total images scanned: {stats['total_scanned']}")
print(f"Duplicates removed:   {stats['duplicates_removed']}")
print(f"\nPer Dataset:")
for ds, count in stats["per_dataset"].items():
    print(f"  {ds}: {count} unique images")
print(f"\nPer Class:")
for cls, info in stats["per_class"].items():
    print(f"  {cls}: {info['total']} total -> {info['train']} train / {info['test']} test")

## 5. Verify Merged Dataset

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

train_dir = os.path.join(MERGED_DIR, "train")
test_dir = os.path.join(MERGED_DIR, "test")

train_counts = {}
test_counts = {}
for cls in CLASSES:
    td = os.path.join(train_dir, cls)
    ed = os.path.join(test_dir, cls)
    train_counts[cls] = len(os.listdir(td)) if os.path.exists(td) else 0
    test_counts[cls] = len(os.listdir(ed)) if os.path.exists(ed) else 0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ["#FF6B6B", "#4ECDC4", "#45B7D1", "#96CEB4"]

axes[0].bar(train_counts.keys(), train_counts.values(), color=colors)
axes[0].set_title("Training Set", fontsize=14, fontweight="bold")
for i, (cls, count) in enumerate(train_counts.items()):
    axes[0].text(i, count + 20, str(count), ha="center", fontweight="bold")

axes[1].bar(test_counts.keys(), test_counts.values(), color=colors)
axes[1].set_title("Test Set", fontsize=14, fontweight="bold")
for i, (cls, count) in enumerate(test_counts.items()):
    axes[1].text(i, count + 5, str(count), ha="center", fontweight="bold")

total = sum(train_counts.values()) + sum(test_counts.values())
plt.suptitle(f"Merged Dataset: {total} Total Images", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(BASE_DIR, "dataset_distribution.png"), dpi=150, bbox_inches="tight")
plt.show()

print(f"\nTotal Training: {sum(train_counts.values())}")
print(f"Total Testing:  {sum(test_counts.values())}")
print(f"Grand Total:    {total}")